#  Chatbot using Hugging Face Transformers


##  Step 1: Install Required Libraries

In [2]:
# Install Hugging Face Transformers library
!pip install transformers torch --quiet

##  Step 2: Import Libraries

In [3]:
# Import required libraries
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer

print("Libraries imported successfully!")
print(f"PyTorch version: {torch.__version__}")

Libraries imported successfully!
PyTorch version: 2.10.0+cu128


##  Step 3: Load Pre-trained Model and Tokenizer

We use **DialoGPT-medium** — a transformer model fine-tuned specifically for multi-turn conversational dialogue.  
It is based on GPT-2 and trained by Microsoft on Reddit conversations.

In [4]:
# Model name from Hugging Face Model Hub
MODEL_NAME = "microsoft/DialoGPT-medium"

print(f"Loading model: {MODEL_NAME} ...")

# Load tokenizer — converts text to tokens the model understands
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

# Load the pre-trained causal language model
model = AutoModelForCausalLM.from_pretrained(MODEL_NAME)

# Set model to evaluation mode (disables dropout layers used only during training)
model.eval()

print("Model and tokenizer loaded successfully!")

Loading model: microsoft/DialoGPT-medium ...


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json:   0%|          | 0.00/642 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/614 [00:00<?, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

...

model.safetensors:   0%|          | 0.00/863M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/293 [00:00<?, ?it/s]

The tied weights mapping and config for this model specifies to tie transformer.wte.weight to lm_head.weight, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning
GPT2LMHeadModel LOAD REPORT from: microsoft/DialoGPT-medium
Key                              | Status     |  | 
---------------------------------+------------+--+-
transformer.h.{0...23}.attn.bias | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


generation_config.json:   0%|          | 0.00/124 [00:00<?, ?B/s]

Model and tokenizer loaded successfully!


##  Step 4: Define Response Generation Function

This function handles the core chatbot logic:
- Tokenizes user input
- Maintains conversation history for context
- Generates a response using the transformer model
- Decodes and returns the response as text

In [5]:
def generate_response(user_input, chat_history_ids=None, max_history_tokens=900):
    """
    Generates a chatbot response using DialoGPT.

    Parameters:
        user_input (str)          : The message typed by the user.
        chat_history_ids (tensor) : Token IDs of the previous conversation turns.
                                    None for the first message.
        max_history_tokens (int)  : Max tokens to keep from conversation history.
                                    Prevents exceeding DialoGPT's 1024-token limit.

    Returns:
        response (str)            : The chatbot's generated response.
        chat_history_ids (tensor) : Updated conversation history including this turn.
    """

    # Step 1: Tokenize user input and append EOS token to signal end of user's turn
    new_input_ids = tokenizer.encode(
        user_input + tokenizer.eos_token,
        return_tensors='pt'       # Return as PyTorch tensor
    )

    # Step 2: Append new input to existing conversation history
    # This gives the model context from previous turns (multi-turn dialogue)
    if chat_history_ids is not None:
        bot_input_ids = torch.cat([chat_history_ids, new_input_ids], dim=-1)
    else:
        # First turn — no previous history exists yet
        bot_input_ids = new_input_ids

    # Step 3: Truncate history if it exceeds the safe token limit
    # DialoGPT has a hard max context window of 1024 tokens
    # Keeping only the last 900 tokens ensures there's always room for a new reply
    if bot_input_ids.shape[-1] > max_history_tokens:
        bot_input_ids = bot_input_ids[:, -max_history_tokens:]

    # Step 4: Generate response using the model
    # no_grad() disables gradient calculation to save memory during inference
    with torch.no_grad():
        chat_history_ids = model.generate(
            bot_input_ids,
            max_length=bot_input_ids.shape[-1] + 200,  # Relative limit: input length + 200 new tokens
            pad_token_id=tokenizer.eos_token_id,        # Use EOS token as padding
            do_sample=True,                             # Enable sampling for varied responses
            top_k=50,                                   # Consider only top-50 tokens at each step
            top_p=0.95,                                 # Nucleus sampling (cumulative probability cutoff)
            temperature=0.75,                           # Controls randomness — lower = more focused
            repetition_penalty=1.3,                     # Penalises repeated words/phrases in output
        )

    # Step 5: Decode only the newly generated tokens (exclude the input prompt)
    response = tokenizer.decode(
        chat_history_ids[:, bot_input_ids.shape[-1]:][0],
        skip_special_tokens=True    # Remove special tokens like EOS from the output
    )

    return response, chat_history_ids

##  Step 5: Run the Interactive Chatbot

The chatbot runs in a continuous loop:  
`User Input → Tokenization → Model Inference → Decode Response → Display → Repeat`  

Type **`exit`** or **`quit`** to end the conversation.

In [7]:
def run_chatbot():
    """
    Main chatbot loop.
    Accepts user input, generates responses using DialoGPT,
    and continues until the user types 'exit' or 'quit'.
    """

    print("=" * 60)
    print("       AI Chatbot — Powered by DialoGPT")
    print("=" * 60)
    print("Chatbot: Hello! I am your AI assistant. How can I help you today?")
    print("(Type 'exit' or 'quit' to end the conversation)")
    print("-" * 60)

    # Initialize conversation history as None for the first turn
    chat_history_ids = None

    # Continuous conversation loop
    while True:

        # Accept user input from console
        user_input = input("You: ").strip()

        # Handle empty input gracefully
        if not user_input:
            print("Chatbot: Please type something so I can respond!")
            continue

        # Check for exit condition — stop chatbot on 'exit' or 'quit'
        if user_input.lower() in ["exit", "quit"]:
            print("-" * 60)
            print("Chatbot: Thank you for chatting with me. Goodbye!")
            print("=" * 60)
            break

        # Generate response from the transformer model
        response, chat_history_ids = generate_response(user_input, chat_history_ids)

        # Fallback in case model returns an empty response
        if not response:
            response = "I'm not sure how to respond to that. Could you rephrase?"

        # Display the chatbot's response
        print(f"Chatbot: {response}")
        print("-" * 60)


# Start the chatbot
run_chatbot()

       AI Chatbot — Powered by DialoGPT
Chatbot: Hello! I am your AI assistant. How can I help you today?
(Type 'exit' or 'quit' to end the conversation)
------------------------------------------------------------
You: hello


The attention mask is not set and cannot be inferred from input because pad token is same as eos token. As a consequence, you may observe unexpected behavior. Please pass your input's `attention_mask` to obtain reliable results.


Chatbot: Hey there! How are you? :D
------------------------------------------------------------
You: What is Artificial Intelligence?
Chatbot: It's a great game. A lot of different things to do with it though
------------------------------------------------------------
You: Who created Python?
Chatbot: A bunch of nerds, probably... I don't know any other examples but they're all awesome and creative too.
------------------------------------------------------------
You: thank you
Chatbot: You betcha
------------------------------------------------------------
You: exit
------------------------------------------------------------
Chatbot: Thank you for chatting with me. Goodbye!
